# Modeling solar energy production

This notebook trains Ridge, Random Forest, and XGBoost regressors to
forecast monthly solar PV production. We use a temporal train/test
split, cross-validate with TimeSeriesSplit, and compare all three models.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

sys.path.insert(0, '..')
from src.data_loader import load_and_prepare_data
from src.model import (
    get_models, temporal_train_test_split, prepare_features,
    compute_metrics, FEATURE_COLUMNS, save_model
)

pd.set_option('display.max_columns', 50)

## 1. Load prepared data

In [ ]:
data = load_and_prepare_data(force_refresh=False)
production = data['production']

print(f"Production records: {len(production)}")
print(f"Facilities: {production['facility_name'].nunique()}")
print(f"Features: {FEATURE_COLUMNS}")

## 2. Temporal train/test split

The most recent 12 months of data are held out for testing, ensuring
no future data leaks into training.

In [ ]:
train_df, test_df = temporal_train_test_split(production, test_months=12)

X_train, y_train, train_clean = prepare_features(train_df)
X_test, y_test, test_clean = prepare_features(test_df)

print(f"Train: {len(X_train)} samples, {train_clean['period_dt'].min().date()} to {train_clean['period_dt'].max().date()}")
print(f"Test:  {len(X_test)} samples, {test_clean['period_dt'].min().date()} to {test_clean['period_dt'].max().date()}")

## 3. Train Ridge regression

In [ ]:
models = get_models()

ridge = models['Ridge']
ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)
ridge_metrics = compute_metrics(y_test.values, ridge_pred)
print(f"Ridge: {ridge_metrics}")

## 4. Train Random Forest

In [ ]:
rf = models['RandomForest']
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_metrics = compute_metrics(y_test.values, rf_pred)
print(f"Random Forest: {rf_metrics}")

## 5. Train XGBoost

In [ ]:
xgb = models.get('XGBoost')
if xgb is not None:
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    xgb_metrics = compute_metrics(y_test.values, xgb_pred)
    print(f"XGBoost: {xgb_metrics}")
else:
    print("XGBoost not available. Install with: pip install xgboost")
    xgb_pred = None
    xgb_metrics = None

## 6. TimeSeriesSplit cross-validation

We use `TimeSeriesSplit` with 5 folds to assess model stability across
different temporal windows.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_results = {}
for name, model_obj in models.items():
    scores = cross_val_score(
        model_obj, X_train, y_train, cv=tscv,
        scoring='r2', n_jobs=-1
    )
    cv_results[name] = scores
    print(f"{name}: mean R2 = {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
cv_df = pd.DataFrame(cv_results)
cv_long = cv_df.melt(var_name='Model', value_name='R2')

fig = px.box(
    cv_long, x='Model', y='R2', points='all',
    title='Cross-validation R2 (TimeSeriesSplit, 5 folds)'
)
fig.update_layout(height=400)
fig.show()

## 7. Compare models on the test set

In [ ]:
comp_rows = [
    {'Model': 'Ridge', **{k: round(v, 4) for k, v in ridge_metrics.items()}},
    {'Model': 'RandomForest', **{k: round(v, 4) for k, v in rf_metrics.items()}},
]
if xgb_metrics is not None:
    comp_rows.append(
        {'Model': 'XGBoost', **{k: round(v, 4) for k, v in xgb_metrics.items()}}
    )

comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

best_name = comp_df.loc[comp_df['R2'].idxmax(), 'Model']
print(f"\nBest model by R2: {best_name}")

In [ ]:
fig = go.Figure()
for metric in ['MAE', 'RMSE']:
    fig.add_trace(go.Bar(
        name=metric, x=comp_df['Model'], y=comp_df[metric]
    ))

fig.update_layout(
    barmode='group',
    title='Model comparison: MAE and RMSE (lower is better)',
    yaxis_title='Error (kWh)', height=400
)
fig.show()

## 8. Save the best model

In [ ]:
best_model = models[best_name]
save_model(best_model, best_name)
print(f"Saved {best_name} model.")

## 9. Summary

Three models were trained on temporal features (cyclical month, lag,
rolling averages) to forecast monthly solar PV production:

- **Ridge** provides a fast linear baseline that captures seasonality
  through the sin/cos encoding.
- **Random Forest** handles non-linear interactions between features.
- **XGBoost** typically achieves the best balance of bias and variance.

TimeSeriesSplit cross-validation confirms performance stability across
different time windows. The next notebook evaluates the best model in
detail, including seasonal error analysis and business context.